# ResNet 깊이와 학습 곡선 / Depth and training error

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/No1Joon/oh-my-notebooks/blob/main/notebooks/resnet_depth_curves.ipynb)

Compare 20 and 56 layers with and without parameter-free shortcuts. Every model sees the same 1,600 CIFAR-10 training images and 500 test images. Eight epochs at batch size 64 give each model 200 updates. The original study used 64,000. See [He et al.](https://arxiv.org/abs/1512.03385) and [Krizhevsky (2009)](https://www.cs.toronto.edu/~kriz/cifar.html). For English charts, set the CHART_LANG environment variable to en before running the final cell. GPU runtime recommended.

## Setup and data

In [ ]:
import hashlib
import json
import os
import pickle
from pathlib import Path
import tarfile
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

SEED = 42
EPOCHS = 8
BATCH = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(2)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
print("PyTorch", torch.__version__, "device", torch.cuda.get_device_name() if DEVICE.type == "cuda" else "CPU")

# Source: https://www.cs.toronto.edu/~kriz/cifar.html (Krizhevsky, 2009).
# The author requests citation and gives no blanket image redistribution license.
# Retrieve the author's archive at runtime. Publish only aggregate errors.
archive = Path("cifar-10-python.tar.gz")
if not archive.exists():
    urlretrieve("https://storage.googleapis.com/tensorflow/tf-keras-datasets/cifar-10-python.tar.gz", archive)
with archive.open("rb") as source:
    digest = hashlib.file_digest(source, "sha256").hexdigest()
    if digest != "6d958be074577803d12ecdefd02955f39262c83c16fe9348329d7fe0b5c001ce":
        raise ValueError("The archive does not match the SHA-256 published by Keras.")

def balanced_sample(rows, count_per_class):
    rng = np.random.default_rng(SEED)
    idx = np.concatenate([
        rng.choice(np.flatnonzero(rows[:, 0] == label), count_per_class, replace=False)
        for label in range(10)
    ])
    pixels = rows[idx, 1:].reshape(-1, 3, 32, 32)
    x = torch.tensor(pixels, device=DEVICE, dtype=torch.float32) / 255
    y = torch.tensor(rows[idx, 0], device=DEVICE, dtype=torch.long)
    return x, y

with tarfile.open(archive, "r:gz") as dataset:
    def read_batch(name):
        with dataset.extractfile(f"cifar-10-batches-py/{name}") as src:
            batch = pickle.load(src, encoding="bytes")
            return np.column_stack((np.asarray(batch[b"labels"], dtype=np.uint8), batch[b"data"]))
    train_rows = np.concatenate([read_batch(f"data_batch_{i}") for i in range(1, 6)])
    train_x, train_y = balanced_sample(train_rows, 160)
    test_x, test_y = balanced_sample(read_batch("test_batch"), 50)
mean = train_x.mean((0, 2, 3), keepdim=True)
std = train_x.std((0, 2, 3), keepdim=True)
train_x, test_x = (train_x - mean) / std, (test_x - mean) / std


## Model: same weights and parameter count within each depth

In [ ]:
class Block(nn.Module):
    def __init__(self, in_channels, out_channels, stride, shortcut):
        super().__init__()
        self.shortcut = shortcut
        self.stride = stride
        self.branch = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        branch = self.branch(x)
        if self.shortcut:
            skip = x[:, :, ::self.stride, ::self.stride]
            missing = branch.size(1) - skip.size(1)
            skip = F.pad(skip, (0, 0, 0, 0, missing // 2, missing - missing // 2))
            branch = branch + skip
        return F.relu(branch)

def make_model(depth, shortcut):
    n = (depth - 2) // 6
    stages = [nn.Conv2d(3, 16, 3, padding=1, bias=False), nn.BatchNorm2d(16), nn.ReLU()]
    previous = 16
    for channels in (16, 32, 64):
        for _ in range(n):
            stages.append(Block(previous, channels, 1 if previous == channels else 2, shortcut))
            previous = channels
    stages.extend([nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 10)])
    model = nn.Sequential(*stages)
    for layer in model.modules():
        if isinstance(layer, nn.Conv2d):
            nn.init.kaiming_normal_(layer.weight, mode="fan_out", nonlinearity="relu")
    return model.to(DEVICE)


## Four matched runs with post-epoch evaluation

In [ ]:
@torch.inference_mode()
def error_percent(model, images, labels):
    model.eval()
    correct = sum((model(x).argmax(1) == y).sum().item()
                  for x, y in zip(images.split(BATCH), labels.split(BATCH)))
    return 100 * (len(labels) - correct) / len(labels)

history = {}
for depth in (20, 56):
    for shortcut in (False, True):
        torch.manual_seed(SEED)
        model = make_model(depth, shortcut)
        order_rng = torch.Generator().manual_seed(SEED)
        optimizer = torch.optim.SGD(model.parameters(), lr=0.03, momentum=0.9, weight_decay=0.0001)
        schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
        name = f"{'residual' if shortcut else 'plain'}-{depth}"
        run = {"parameters": sum(p.numel() for p in model.parameters()), "train": [], "test": []}
        for epoch in range(EPOCHS):
            model.train()
            for idx in torch.randperm(len(train_y), generator=order_rng).split(BATCH):
                optimizer.zero_grad(set_to_none=True)
                loss = F.cross_entropy(model(train_x[idx]), train_y[idx])
                loss.backward()
                optimizer.step()
            schedule.step()
            run["train"].append(error_percent(model, train_x, train_y))
            run["test"].append(error_percent(model, test_x, test_y))
        history[name] = run
        print(name, json.dumps(run), flush=True)
    assert history[f"plain-{depth}"]["parameters"] == history[f"residual-{depth}"]["parameters"]


## Draw both depths for each family, using the same measured history

In [ ]:
requested = os.getenv("CHART_LANG", "ko")
if requested not in {"ko", "en", "both"}:
    raise ValueError("CHART_LANG accepts ko, en or both")
for lang in (["ko", "en"] if requested == "both" else [requested]):
    ko = lang == "ko"
    family = "DejaVu Sans"
    if ko:
        path = Path("NanumGothic-Regular.ttf")
        if not path.exists():
            urlretrieve("https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf", path)
        font_manager.fontManager.addfont(path)
        family = font_manager.FontProperties(fname=str(path)).get_name()
    with plt.rc_context({"font.family": family, "font.size": 12, "axes.spines.top": False, "axes.spines.right": False}):
        for key, label in [("plain", "일반망" if ko else "Plain network"), ("residual", "잔차망" if ko else "Residual network")]:
            fig, ax = plt.subplots(figsize=(8.4, 4.2), layout="constrained")
            for depth, color in [(20, "#235996"), (56, "#c15721")]:
                ax.plot(range(1, EPOCHS + 1), history[f"{key}-{depth}"]["train"], marker="o", color=color,
                        label=f"{depth}층" if ko else f"{depth} layers")
            ax.set(title=f"{label}: 학습 세트 오분류율" if ko else f"{label}: training-set classification error",
                   xlabel="학습 횟수 (epoch)" if ko else "Epoch",
                   ylabel="오분류율 (%)" if ko else "Classification error (%)",
                   ylim=(0, 100), xticks=range(1, EPOCHS + 1))
            ax.grid(axis="y", alpha=.18)
            ax.legend()
            plt.show()
            plt.close(fig)
